# Análise de Entropia das Saídas Criptográficas

In [ ]:
# !pip install scipy matplotlib numpy mnemonic eth-keys

In [ ]:
import hashlib
import hmac as hmac_lib
import math
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# BIP-39
try:
    from mnemonic import Mnemonic
    BIP39_OK = True
except ImportError:
    BIP39_OK = False
    print("[aviso] mnemonic não instalado — célula BIP-39 será ignorada.")
    print("        Execute: pip install mnemonic")

# Dependências adicionais
from scipy.stats import chi2, chisquare
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from collections import Counter
import hashlib, hmac as hmac_lib

## Metodos

In [ ]:
# Calcula a entropia de Shannon em bits/byte sobre a distribuição
def entropia_bytes(data: bytes) -> float:
    
    n = len(data)
    if n == 0:
        return 0.0
    freq = Counter(data)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())


# número total de bits no dado
def bits_totais(data: bytes) -> int:
    return len(data) * 8


print(f"Entropia de zeros:         {entropia_bytes(bytes(32)):.4f} bits/byte")
print(f"Entropia de bytes aleat.:  {entropia_bytes(bytes(range(256))*4):.4f} bits/byte")

# Entropia por posição de byte
def entropia_coluna(col: np.ndarray) -> float:
    """Entropia de Shannon (bits) de um array de bytes (valores 0-255)."""
    n = len(col)
    freq = Counter(col.tolist())
    return -sum((c / n) * np.log2(c / n) for c in freq.values())

## Simulação

In [ ]:
SENHA      = b"senha_fraca"
SALT       = b"salt_fixo_aula"
CHAVE_HMAC = b"chave_hmac_aula"

N_HASH = 5   # iterações para Hash, HMAC e Salt  
N_KDF  = 600     # iterações para KDFs               
N_PBKDF2 = N_KDF  #200  
N_ANALISE = 200  # saídas por método — aumentar para mais precisão       

In [ ]:
# Gerar N_ANALISE
print(f"Gerando {N_ANALISE:,} saídas por método...")

metodos = {
    "SHA-256":     lambda i: hashlib.sha256(SENHA + str(i).encode()).digest(),
    "SHA3-256":    lambda i: hashlib.sha3_256(SENHA + str(i).encode()).digest(),
    "HMAC-SHA256": lambda i: hmac_lib.new(CHAVE_HMAC, SENHA + str(i).encode(), hashlib.sha256).digest(),
    "SHA256+Salt": lambda i: hashlib.sha256(SALT + SENHA + str(i).encode()).digest(),
    "PBKDF2":      lambda i: hashlib.pbkdf2_hmac("sha256", SENHA + str(i).encode(), SALT, 100_000, dklen=32),
}



saidas = {}
for nome, fn in metodos.items():
    n = N_PBKDF2 if nome == "PBKDF2" else N_ANALISE
    print(f"  {nome} ({n:,} amostras)...", end=" ", flush=True)
    # matriz shape (n, digest_size) — cada linha é um digest
    buf = np.frombuffer(
        b"".join(fn(i) for i in range(n)),
        dtype=np.uint8
    ).reshape(n, -1)
    saidas[nome] = buf
    print("ok")

print("Pronto!")

## Entropia por Byte

In [ ]:
fig, axes = plt.subplots(len(saidas), 1,
                         figsize=(14, 2.8 * len(saidas)),
                         sharex=False)
fig.suptitle("Entropia por Posição de Byte", fontsize=14, fontweight="bold", y=1.01)

CORES_MET = {
    "SHA-256":     "#3498db",
    "SHA3-256":    "#9b59b6",
    "HMAC-SHA256": "#8e44ad",
    "SHA256+Salt": "#27ae60",
    "PBKDF2":      "#f39c12",
}

entropias_por_pos = {}  # guardamos para uso posterior

for ax, (nome, mat) in zip(axes, saidas.items()):
    n_pos = mat.shape[1]
    ents  = np.array([entropia_coluna(mat[:, j]) for j in range(n_pos)])
    entropias_por_pos[nome] = ents

    ax.bar(range(n_pos), ents,
           color=CORES_MET[nome], edgecolor="none", alpha=0.85)
    ax.axhline(8.0, color="black",  linestyle="--", linewidth=1.0,
               label="Máximo (8 bits)")
    ax.axhline(np.mean(ents), color="red", linestyle=":",
               linewidth=1.2, label=f"Média = {np.mean(ents):.4f} bits")
    ax.set_ylim(6.5, 8.3)
    ax.set_ylabel("bits/byte", fontsize=9)
    ax.set_title(nome, fontsize=11, color=CORES_MET[nome], fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(4))
    ax.set_xlabel("Posição do byte no digest", fontsize=9)

plt.tight_layout()
plt.savefig("entropia_por_posicao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: entropia_por_posicao.png")

## Teste Qui-Quadrado por posição

$H_0$: a distribuição dos bytes na posição j é uniforme (256 valores equiprováveis).

```
Se p-valor < 0.05  
    rejeitamos H0 # posição com distribuição enviesada.
```
Esperado: nenhuma posição deve rejeitar $H_0$ para boas funções criptográficas.

In [ ]:
alpha = 0.05
resultados_chi2 = {}

print(f"{'Método':<14} {'Posições testadas':>18} {'Rejeições (p<{:.2f})'.format(alpha):>22} {'% rejeições':>12}")
print("-" * 70)

for nome, mat in saidas.items():
    n_pos      = mat.shape[1]
    n_amostras = mat.shape[0]
    esperado   = np.full(256, n_amostras / 256)   # frequência esperada uniforme

    p_valores   = []
    rejeicoes   = 0
    for j in range(n_pos):
        obs = np.bincount(mat[:, j], minlength=256).astype(float)
        _, p = chisquare(obs, f_exp=esperado)
        p_valores.append(p)
        if p < alpha:
            rejeicoes += 1

    resultados_chi2[nome] = np.array(p_valores)
    pct = 100 * rejeicoes / n_pos
    print(f"{nome:<14} {n_pos:>18} {rejeicoes:>22} {pct:>11.1f}%")

print()
print("Interpretação: % de rejeições próxima de α=5% indica distribuição uniforme")
print("(rejeições aleatórias esperadas pelo próprio teste).")
print("% muito acima de 5% indica enviesamento — sinal de fraqueza.")

## Distribuição de bytes por posição (Heatmap)

Para cada método, plotamos um heatmap (posição × valor_byte).
Uma boa função hash deve mostrar cor uniforme em todo o mapa.
Concentrações de cor indicam enviesamento.

In [ ]:
fig, axes = plt.subplots(1, len(saidas),
                         figsize=(5 * len(saidas), 6))
fig.suptitle("Heatmap: Distribuição de Bytes por Posição",
             fontsize=14, fontweight="bold")

for ax, (nome, mat) in zip(axes, saidas.items()):
    n_pos = mat.shape[1]
    # matriz (256, n_pos) | frequência de cada valor em cada posição
    hmap = np.zeros((256, n_pos), dtype=float)
    for j in range(n_pos):
        counts = np.bincount(mat[:, j], minlength=256)
        hmap[:, j] = counts / counts.sum()   # normalizacao das probs

    im = ax.imshow(hmap, aspect="auto", cmap="Blues",
                   vmin=0, vmax=2/256,   # realça desvios da uniforme (1/256)
                   origin="lower")
    ax.set_title(nome, fontsize=10, fontweight="bold",
                 color=CORES_MET[nome])
    ax.set_xlabel("Posição (byte)", fontsize=8)
    ax.set_ylabel("Valor do byte (0–255)", fontsize=8)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(64))
    ax.xaxis.set_major_locator(mticker.MultipleLocator(8))

# Barra de cor compartilhada
fig.colorbar(im, ax=axes, orientation="vertical",
             fraction=0.015, pad=0.04,
             label="P(valor | posição) — uniforme ≈ 1/256")

plt.tight_layout()
plt.savefig("heatmap_bytes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: heatmap_bytes.png")

In [ ]:
print(f"{'Método':<14} {'H média':>10} {'H mín':>8} {'H máx':>8} {'% p<0.05':>10}  Qualidade")
print("─" * 65)

for nome in saidas:
    ents  = entropias_por_pos[nome]
    pvals = resultados_chi2[nome]
    pct_rej = 100 * (pvals < alpha).sum() / len(pvals)

    # Heurística de qualidade
    if ents.mean() >= 7.95 and pct_rej <= 10:
        qualidade = " Excelente"
    elif ents.mean() >= 7.80 and pct_rej <= 15:
        qualidade = " Bom"
    else:
        qualidade = " Atenção"

    print(f"{nome:<14} {ents.mean():>10.4f} {ents.min():>8.4f} "
          f"{ents.max():>8.4f} {pct_rej:>9.1f}%  {qualidade}")

print()
print("Nota: % rejeições χ² esperada ≈ 5% (falsos positivos do próprio teste).")